# 06 오답 분석 및 최종 모델 선택

- 05번에서 저장한 baseline, ablation, proposed 모델을 같은 validation/test split에서 다시 평가한다.
- validation F1을 기준으로 최종 모델을 선택하고, 선택 이후 test 데이터에서 FP/FN 오답 패턴을 확인한다.
- 06번은 별점 정제 단계가 아니라 모델 선택과 진단 단계이다. 
- 성능 기준 선택 모델은 `outputs/final_selected_model.joblib`로 저장한다.
- 프로젝트 제안 모델은 `outputs/final_proposed_model.joblib`로 따로 저장해 07번에서 함께 비교한다.

## 1. 라이브러리 로드

- 저장된 모델 설정값을 불러오고 validation/test split 기준 성능과 오답 데이터를 다시 계산하기 위한 라이브러리를 로드한다. 
- 05번과 같은 split을 재현해야 각 모델을 같은 조건에서 비교할 수 있다.


In [ ]:
import os

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

## 2. Validation/Test 데이터와 원본 리뷰 연결

05번과 같은 방식으로 train/validation/test split을 다시 만든다. 예측 결과를 실제 리뷰 원문과 연결할 수 있도록 validation/test index를 기준으로 원본 리뷰 행도 함께 복원한다.

- validation: 최종 모델 선택에 사용하는 데이터
- test: 선택이 끝난 뒤 참고 성능과 오답을 확인하는 데이터

이 연결을 만들어 두면 FP/FN 샘플을 볼 때 리뷰 텍스트, 별점, 사진 수, 글자 수, 이모지 수를 함께 확인할 수 있다.


In [ ]:
SEED = 42
LABEL_COL = 'label'
TEXT_COL = 'cleaned_review_text'
META_COLS = ['text_length', 'emoji_count', 'photo_count']
RAW_META_COLS = ['text_length', 'emoji_count', 'photo_count']

raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')

emb_cols = [f'kcbert_{i}' for i in range(768)]
meta_cols = META_COLS
hybrid_cols = emb_cols + meta_cols

X_all = raw_df[hybrid_cols].copy()
y_all = raw_df[LABEL_COL].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all,
    y_all,
    test_size=0.3,
    random_state=SEED,
    stratify=y_all,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp,
)

val_review_df = raw_df.loc[X_val.index].copy().reset_index().rename(columns={'index': 'raw_index'})
test_review_df = raw_df.loc[X_test.index].copy().reset_index().rename(columns={'index': 'raw_index'})

text_val = val_review_df[TEXT_COL].fillna('').astype(str)
text_test = test_review_df[TEXT_COL].fillna('').astype(str)

print('validation feature shape:', X_val.shape)
print('test feature shape:', X_test.shape)
print('validation review shape:', val_review_df.shape)
print('test review shape:', test_review_df.shape)
print('validation label 분포:', y_val.value_counts().sort_index().to_dict())
print('test label 분포:', y_test.value_counts().sort_index().to_dict())
print('excluded feature:', ['rating'])

## 3. 저장된 모델 bundle 로드

05번에서 저장한 `outputs/baseline_model_registry.csv`를 읽어 평가 후보 모델을 구성한다. 이 registry에는 TF-IDF + RandomForest, KcBERT-only, metadata-only, metadata subset ablation, proposed hybrid 모델이 모두 들어 있다.

각 bundle에는 학습된 모델, 입력 feature 정보, validation에서 조정한 threshold가 들어 있다. 06번은 이 값을 그대로 사용해 후보 모델을 같은 split에서 재평가한다.


In [ ]:
MODEL_REGISTRY_PATH = 'outputs/baseline_model_registry.csv'
if not os.path.exists(MODEL_REGISTRY_PATH):
    raise FileNotFoundError('outputs/baseline_model_registry.csv가 없습니다. 05번 노트북을 먼저 실행하세요.')

model_registry = pd.read_csv(MODEL_REGISTRY_PATH)
required_registry_cols = {'model', 'feature_set', 'path', 'input_type'}
missing_registry_cols = required_registry_cols - set(model_registry.columns)
if missing_registry_cols:
    raise ValueError(f'모델 registry 필수 컬럼이 없습니다: {sorted(missing_registry_cols)}')

model_specs = model_registry[['model', 'feature_set', 'path', 'input_type']].to_dict(orient='records')
missing_paths = [spec['path'] for spec in model_specs if not os.path.exists(spec['path'])]
if missing_paths:
    raise FileNotFoundError(f'모델 파일이 없습니다: {missing_paths}')

model_bundles = {spec['model']: joblib.load(spec['path']) for spec in model_specs}

loaded_models = pd.DataFrame([
    {'model': spec['model'], 'feature_set': spec['feature_set'], 'path': spec['path'], 'input_type': spec['input_type']}
    for spec in model_specs
])
display(loaded_models)


## 4. Validation/Test 성능 비교

모든 저장 모델을 같은 validation/test split에 적용한다. 텍스트 기반 모델은 정제 텍스트를 입력으로 사용하고, tabular 모델은 bundle에 저장된 `feature_cols`만 사용한다.

- validation 성능은 최종 모델 선택에 사용한다.
- test 성능은 모델 선택 이후 참고 성능으로만 확인한다.
- 각 모델은 저장된 threshold를 사용해 확률을 0/1 예측으로 변환한다.

validation과 test의 역할을 분리하면 test 성능을 보고 모델을 고르는 데이터 누수를 줄일 수 있다.


In [ ]:
def predict_prob(spec, bundle, split):
    model = bundle['model']

    if split == 'validation':
        text_data = text_val
        X_data = X_val
    elif split == 'test':
        text_data = text_test
        X_data = X_test
    else:
        raise ValueError(f'Unsupported split: {split}')

    if spec['input_type'] == 'text':
        return model.predict_proba(text_data)[:, 1]

    feature_cols = bundle['feature_cols']
    return model.predict_proba(X_data[feature_cols])[:, 1]


def metric_row(spec, bundle, split, y_true, prob):
    threshold = float(bundle.get('best_threshold', bundle.get('default_threshold', 0.5)))
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    return {
        'model': spec['model'],
        'feature_set': spec['feature_set'],
        'split': split,
        'threshold': threshold,
        'accuracy': float(accuracy_score(y_true, pred)),
        'f1': float(f1_score(y_true, pred)),
        'pr_auc': float(average_precision_score(y_true, prob)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, prob)),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    }


model_probabilities = {'validation': {}, 'test': {}}
rows = []
for spec in model_specs:
    bundle = model_bundles[spec['model']]

    val_prob = predict_prob(spec, bundle, split='validation')
    test_prob = predict_prob(spec, bundle, split='test')

    model_probabilities['validation'][spec['model']] = val_prob
    model_probabilities['test'][spec['model']] = test_prob

    rows.append(metric_row(spec, bundle, 'validation', y_val, val_prob))
    rows.append(metric_row(spec, bundle, 'test', y_test, test_prob))

comparison_all = pd.DataFrame(rows)
validation_comparison = (
    comparison_all[comparison_all['split'] == 'validation']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
validation_comparison.insert(0, 'rank', validation_comparison.index + 1)

validation_accuracy_comparison = (
    comparison_all[comparison_all['split'] == 'validation']
    .sort_values(['accuracy', 'f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
validation_accuracy_comparison.insert(0, 'rank_accuracy', validation_accuracy_comparison.index + 1)

test_comparison = (
    comparison_all[comparison_all['split'] == 'test']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
test_comparison.insert(0, 'rank', test_comparison.index + 1)

round_cols = {
    'threshold': 4,
    'accuracy': 4,
    'f1': 4,
    'pr_auc': 4,
    'precision': 4,
    'recall': 4,
    'roc_auc': 4,
}

print('Validation F1 기준 모델 선택 순위')
display(validation_comparison.round(round_cols))

print('Validation accuracy 기준 참고 순위')
display(validation_accuracy_comparison.round(round_cols))

print('Test 참고 성능 순위')
display(test_comparison.round(round_cols))


### 성능 비교표 컬럼 설명

- `rank`: 해당 split 안에서의 모델 순위이다. `validation`에서는 최종 모델 선택 순위로 사용하고, `test`에서는 참고 성능 순위로만 확인한다.
- `model`: 평가한 모델 이름이다.
- `feature_set`: 모델이 사용한 입력 feature 종류이다. 예를 들어 TF-IDF 텍스트, KcBERT PCA, 메타데이터, KcBERT PCA + 메타데이터를 의미한다.
- `split`: 평가에 사용한 데이터 구분이다. `validation`은 모델 선택용, `test`는 최종 참고 성능 확인용이다.
- `threshold`: 이벤트 확률을 0/1 예측으로 바꾸는 기준값이다. `event_prob >= threshold`이면 이벤트 리뷰로 예측한다.
- `accuracy`: 전체 리뷰 중 맞춘 비율이다. 클래스 불균형의 영향을 받을 수 있으므로 참고 지표로만 본다.
- `f1`: 이벤트 리뷰 클래스에 대한 F1-score이다. precision과 recall의 균형을 보는 지표이며, 이 노트북에서는 validation F1을 최종 모델 선택 기준으로 사용한다.
- `pr_auc`: Precision-Recall 곡선 아래 면적이다. 이벤트 리뷰처럼 클래스 불균형이 있는 문제에서 모델의 확률 예측 품질을 참고하기 위해 사용한다.
- `precision`: 이벤트 리뷰라고 예측한 것 중 실제 이벤트 리뷰의 비율이다. 낮으면 일반 리뷰를 이벤트 리뷰로 오탐하는 FP가 많다는 뜻이다.
- `recall`: 실제 이벤트 리뷰 중 모델이 이벤트 리뷰로 잡아낸 비율이다. 낮으면 이벤트 리뷰를 일반 리뷰로 놓치는 FN이 많다는 뜻이다.
- `roc_auc`: ROC 곡선 아래 면적이다. threshold와 무관하게 일반 리뷰와 이벤트 리뷰를 구분하는 전반적인 확률 순위 성능을 참고한다.
- `tn`: 실제 일반 리뷰를 일반 리뷰로 맞춘 개수이다.
- `fp`: 실제 일반 리뷰를 이벤트 리뷰로 잘못 예측한 개수이다.
- `fn`: 실제 이벤트 리뷰를 일반 리뷰로 잘못 예측한 개수이다.
- `tp`: 실제 이벤트 리뷰를 이벤트 리뷰로 맞춘 개수이다.

## 5. 최종 모델 선택

- 이벤트 리뷰 탐지가 목적이므로 validation F1을 1차 기준으로 최종 모델을 선택한다. 
- F1은 precision과 recall의 균형을 반영하므로, 이벤트 리뷰를 너무 많이 놓치거나 과도하게 오탐하는 모델을 피하는 데 적합하다.
- PR-AUC와 precision/recall은 보조 해석 지표로 사용한다. 선택된 모델의 test 성능은 기록하지만, test 결과를 보고 모델을 바꾸지는 않는다.
- 이 기준을 고정해 두면 최종 모델 선택 과정이 일관되고, test 데이터는 일반화 성능 확인용으로 남길 수 있다.


In [ ]:
selected_row = validation_comparison.iloc[0].copy()
selected_model_name = selected_row['model']
selected_spec = next(spec for spec in model_specs if spec['model'] == selected_model_name)
selected_bundle = model_bundles[selected_model_name]
selected_threshold = float(selected_bundle.get('best_threshold', selected_bundle.get('default_threshold', 0.5)))
selected_test_row = test_comparison[test_comparison['model'] == selected_model_name].iloc[0].copy()

proposed_model_name = 'proposed_hybrid_mlp_04'
proposed_spec = next(spec for spec in model_specs if spec['model'] == proposed_model_name)
proposed_bundle = model_bundles[proposed_model_name]
proposed_threshold = float(proposed_bundle.get('best_threshold', proposed_bundle.get('default_threshold', 0.5)))
proposed_row = validation_comparison[validation_comparison['model'] == proposed_model_name].iloc[0].copy()
proposed_test_row = test_comparison[test_comparison['model'] == proposed_model_name].iloc[0].copy()

selection_reason = (
    '저장된 threshold를 적용했을 때 validation 이벤트 F1이 가장 높아 선택했다. '
    'test 데이터는 최종 참고 성능과 오답 분석에만 사용했다.'
)
proposed_reason = '프로젝트 제안 구조인 KcBERT PCA + metadata hybrid 모델이므로 성능 기준 선택 모델과 별도로 저장한다.'

selection_and_proposed_display = pd.DataFrame([
    {
        'role': '성능 기준 선택 모델',
        'model': selected_model_name,
        'feature_set': selected_row['feature_set'],
        'threshold': selected_threshold,
        'validation_rank': int(selected_row['rank']),
        'validation_accuracy': selected_row['accuracy'],
        'validation_f1': selected_row['f1'],
        'validation_pr_auc': selected_row['pr_auc'],
        'validation_precision': selected_row['precision'],
        'validation_recall': selected_row['recall'],
        'test_accuracy_reference': selected_test_row['accuracy'],
        'test_f1_reference': selected_test_row['f1'],
        'test_pr_auc_reference': selected_test_row['pr_auc'],
        'test_precision_reference': selected_test_row['precision'],
        'test_recall_reference': selected_test_row['recall'],
        'test_fp_reference': int(selected_test_row['fp']),
        'test_fn_reference': int(selected_test_row['fn']),
        'note': selection_reason,
    },
    {
        'role': '프로젝트 제안 모델',
        'model': proposed_model_name,
        'feature_set': proposed_row['feature_set'],
        'threshold': proposed_threshold,
        'validation_rank': int(proposed_row['rank']),
        'validation_accuracy': proposed_row['accuracy'],
        'validation_f1': proposed_row['f1'],
        'validation_pr_auc': proposed_row['pr_auc'],
        'validation_precision': proposed_row['precision'],
        'validation_recall': proposed_row['recall'],
        'test_accuracy_reference': proposed_test_row['accuracy'],
        'test_f1_reference': proposed_test_row['f1'],
        'test_pr_auc_reference': proposed_test_row['pr_auc'],
        'test_precision_reference': proposed_test_row['precision'],
        'test_recall_reference': proposed_test_row['recall'],
        'test_fp_reference': int(proposed_test_row['fp']),
        'test_fn_reference': int(proposed_test_row['fn']),
        'note': proposed_reason,
    },
]).round({
    'threshold': 4,
    'validation_accuracy': 4,
    'validation_f1': 4,
    'validation_pr_auc': 4,
    'validation_precision': 4,
    'validation_recall': 4,
    'test_accuracy_reference': 4,
    'test_f1_reference': 4,
    'test_pr_auc_reference': 4,
    'test_precision_reference': 4,
    'test_recall_reference': 4,
})

display(selection_and_proposed_display)

## 6. 선택 모델과 제안 모델 FP/FN 오답 추출

선택 모델과 제안 모델을 test split에 적용한 뒤 오답을 FP와 FN으로 나눈다.

- FP: 실제 일반 리뷰(0)를 이벤트 리뷰(1)로 잘못 예측한 경우 - 오탐
- FN: 실제 이벤트 리뷰(1)를 일반 리뷰(0)로 놓친 경우 - 미탐

FP가 많으면 일반 리뷰까지 이벤트 리뷰로 제외할 위험이 있고, FN이 많으면 이벤트 리뷰가 별점 정제에 그대로 남을 위험이 있다. 따라서 선택 모델과 제안 모델의 두 오류 유형을 나란히 확인한다.


In [ ]:
def make_error_detail(model_name, model_role, model_label, threshold):
    prob = model_probabilities['test'][model_name]
    pred = (prob >= threshold).astype(int)
    frame = test_review_df.copy()
    frame['model'] = model_name
    frame['model_role'] = model_role
    frame['model_label'] = model_label
    frame['threshold'] = threshold
    frame['actual_label'] = y_test.to_numpy()
    frame['pred_label'] = pred
    frame['event_prob'] = prob
    frame['error_type'] = np.select(
        [
            (frame['actual_label'] == 0) & (frame['pred_label'] == 1),
            (frame['actual_label'] == 1) & (frame['pred_label'] == 0),
        ],
        ['FP_일반을_이벤트로_오탐', 'FN_이벤트를_일반으로_미탐'],
        default='correct',
    )
    return frame


selected_error_df = make_error_detail(
    selected_model_name,
    'selected',
    f'선택 모델 ({selected_model_name})',
    selected_threshold,
)
proposed_error_df = make_error_detail(
    proposed_model_name,
    'proposed',
    f'제안 모델 ({proposed_model_name})',
    proposed_threshold,
)
error_detail = pd.concat([selected_error_df, proposed_error_df], ignore_index=True)
error_df = selected_error_df.copy()

summary_cols = ['photo_count', 'text_length', 'emoji_count']
error_summary = (
    error_detail.groupby(['model_role', 'model', 'error_type'])
    .agg(
        count=('error_type', 'size'),
        mean_event_prob=('event_prob', 'mean'),
        mean_photo_count=('photo_count', 'mean'),
        mean_text_length=('text_length', 'mean'),
        mean_emoji_count=('emoji_count', 'mean'),
    )
    .reset_index()
    .sort_values('count', ascending=False)
    .round(4)
)

display(error_summary)

## 7. 오답 샘플 출력

선택 모델과 제안 모델의 FP/FN 샘플을 각각 출력한다. 실제 리뷰 문장을 함께 보면서 이벤트 표현이 애매한 경우, 짧은 리뷰, 사진 수가 많은 리뷰, 이모지가 많은 리뷰처럼 숫자 지표만으로는 보이지 않는 실패 패턴을 확인한다.


In [ ]:
review_cols = [
    'model_role',
    'model',
    'model_label',
    'raw_index',
    'store_name',
    'rating',
    'review_text',
    'cleaned_review_text',
    'photo_count',
    'text_length',
    'emoji_count',
    'actual_label',
    'pred_label',
    'event_prob',
    'error_type',
]
review_cols = [col for col in review_cols if col in error_detail.columns]

fp_samples = (
    error_detail[error_detail['error_type'] == 'FP_일반을_이벤트로_오탐']
    .sort_values(['model_role', 'event_prob'], ascending=[True, False])
    .groupby(['model_role', 'model'], group_keys=False)
    .head(15)
    [review_cols]
)

fn_samples = (
    error_detail[error_detail['error_type'] == 'FN_이벤트를_일반으로_미탐']
    .sort_values(['model_role', 'event_prob'], ascending=[True, True])
    .groupby(['model_role', 'model'], group_keys=False)
    .head(15)
    [review_cols]
)

print('선택 모델/제안 모델 FP 샘플: 일반 리뷰인데 이벤트 리뷰로 예측한 사례')
display(fp_samples)

print('선택 모델/제안 모델 FN 샘플: 이벤트 리뷰인데 일반 리뷰로 놓친 사례')
display(fn_samples)


## 8. 메타데이터 중요도와 행동 패턴 점검

선택 모델과 제안 모델이 tabular feature를 직접 사용하는 경우 test split에서 permutation importance를 계산한다. 특정 메타데이터 컬럼을 섞었을 때 F1이 크게 떨어지면, 해당 컬럼이 모델 판단에 더 큰 영향을 준 것으로 해석한다.

텍스트 기반 모델처럼 메타데이터 컬럼을 직접 쓰지 않는 경우에는 중요도 계산을 건너뛴다. 대신 실제 라벨별로 사진 수, 글자 수, 이모지 수 평균을 모델별로 확인해 행동 패턴 차이를 참고한다.


In [ ]:
def permutation_importance_for_tabular_model(model, X, y_true, threshold, columns, repeats=10, random_state=SEED):
    rng = np.random.default_rng(random_state)
    base_prob = model.predict_proba(X)[:, 1]
    base_pred = (base_prob >= threshold).astype(int)
    base_f1 = f1_score(y_true, base_pred)

    rows = []
    for col in columns:
        drops = []
        for _ in range(repeats):
            X_perm = X.copy()
            shuffled = X_perm[col].to_numpy().copy()
            rng.shuffle(shuffled)
            X_perm[col] = shuffled
            perm_prob = model.predict_proba(X_perm)[:, 1]
            perm_pred = (perm_prob >= threshold).astype(int)
            drops.append(base_f1 - f1_score(y_true, perm_pred))

        rows.append({
            'metadata': col,
            'base_f1': float(base_f1),
            'mean_f1_drop': float(np.mean(drops)),
            'std_f1_drop': float(np.std(drops)),
            'repeats': repeats,
        })
    return pd.DataFrame(rows).sort_values('mean_f1_drop', ascending=False).reset_index(drop=True)


def metadata_importance_for_model(model_name, model_role, model_label, spec, bundle, threshold):
    feature_cols = bundle.get('feature_cols')
    if spec['input_type'] != 'tabular' or feature_cols is None:
        return pd.DataFrame()

    importance_cols = [col for col in meta_cols if col in feature_cols]
    if not importance_cols:
        return pd.DataFrame()

    importance = permutation_importance_for_tabular_model(
        bundle['model'],
        X_test[feature_cols],
        y_test,
        threshold,
        importance_cols,
        repeats=10,
    )
    importance.insert(0, 'model_label', model_label)
    importance.insert(0, 'model_role', model_role)
    importance.insert(0, 'model', model_name)
    return importance


selected_importance = metadata_importance_for_model(
    selected_model_name,
    'selected',
    f'선택 모델 ({selected_model_name})',
    selected_spec,
    selected_bundle,
    selected_threshold,
)
proposed_importance = metadata_importance_for_model(
    proposed_model_name,
    'proposed',
    f'제안 모델 ({proposed_model_name})',
    proposed_spec,
    proposed_bundle,
    proposed_threshold,
)

metadata_importance = pd.concat([selected_importance, proposed_importance], ignore_index=True)
if metadata_importance.empty:
    print('선택 모델과 제안 모델이 메타데이터 컬럼을 직접 사용하지 않아 permutation importance를 건너뜁니다.')
else:
    metadata_importance_display = (
        metadata_importance
        .sort_values(['model_role', 'mean_f1_drop'], ascending=[True, False])
        .reset_index(drop=True)
        .round({
            'base_f1': 4,
            'mean_f1_drop': 4,
            'std_f1_drop': 4,
        })
    )
    display(metadata_importance_display)

behavior_summary = (
    error_detail.groupby(['model_role', 'model', 'actual_label'])
    .agg(
        count=('actual_label', 'size'),
        mean_photo_count=('photo_count', 'mean'),
        mean_text_length=('text_length', 'mean'),
        mean_emoji_count=('emoji_count', 'mean'),
    )
    .reset_index()
    .replace({'actual_label': {0: '일반_리뷰_0', 1: '이벤트_리뷰_1'}})
    .round(4)
)

print('실제 라벨별 메타데이터 평균')
display(behavior_summary)

## 9. KcBERT 오답과 메타데이터 보완 효과 점검

PDF의 오답 데이터 해석 계획에 맞춰 KcBERT-only 모델이 틀린 test 리뷰를 별도로 추출한다. 이후 proposed hybrid 모델이 그중 어떤 사례를 맞췄는지 비교해, 텍스트 임베딩만으로 어려웠던 사례에서 메타데이터 결합이 도움이 되었는지 확인한다.

은어, 신조어, 우회 표현 여부는 자동으로 확정하기 어렵기 때문에 이 셀에서는 사람이 해석할 수 있도록 리뷰 원문, 정제 텍스트, 사진 수, 글자 수, 이모지 수, 각 모델의 이벤트 확률을 함께 출력한다. 사진은 이미지 내용을 분석하지 않고 `photo_count`만 사용한다.


In [ ]:
kcbert_only_model = 'ablation_mlp_kcbert_pca_only'
hybrid_model = proposed_model_name


def get_threshold(model_name):
    bundle = model_bundles[model_name]
    return float(bundle.get('best_threshold', bundle.get('default_threshold', 0.5)))


def get_model_pred(model_name):
    threshold = get_threshold(model_name)
    prob = model_probabilities['test'][model_name]
    pred = (prob >= threshold).astype(int)
    return prob, pred, threshold


kcbert_prob, kcbert_pred, kcbert_threshold = get_model_pred(kcbert_only_model)
hybrid_prob, hybrid_pred, hybrid_threshold = get_model_pred(hybrid_model)

kcbert_error_df = test_review_df.copy()
kcbert_error_df['actual_label'] = y_test.to_numpy()
kcbert_error_df['kcbert_event_prob'] = kcbert_prob
kcbert_error_df['kcbert_pred'] = kcbert_pred
kcbert_error_df['hybrid_event_prob'] = hybrid_prob
kcbert_error_df['hybrid_pred'] = hybrid_pred
kcbert_error_df['kcbert_correct'] = kcbert_error_df['kcbert_pred'] == kcbert_error_df['actual_label']
kcbert_error_df['hybrid_correct'] = kcbert_error_df['hybrid_pred'] == kcbert_error_df['actual_label']
kcbert_error_df['kcbert_error_type'] = np.select(
    [
        (kcbert_error_df['actual_label'] == 0) & (kcbert_error_df['kcbert_pred'] == 1),
        (kcbert_error_df['actual_label'] == 1) & (kcbert_error_df['kcbert_pred'] == 0),
    ],
    ['KcBERT_FP_일반을_이벤트로_오탐', 'KcBERT_FN_이벤트를_일반으로_미탐'],
    default='KcBERT_correct',
)
kcbert_error_df['hybrid_error_type'] = np.select(
    [
        (kcbert_error_df['actual_label'] == 0) & (kcbert_error_df['hybrid_pred'] == 1),
        (kcbert_error_df['actual_label'] == 1) & (kcbert_error_df['hybrid_pred'] == 0),
    ],
    ['Hybrid_FP_일반을_이벤트로_오탐', 'Hybrid_FN_이벤트를_일반으로_미탐'],
    default='Hybrid_correct',
)

kcbert_wrong_mask = ~kcbert_error_df['kcbert_correct']
hybrid_recovered_mask = kcbert_wrong_mask & kcbert_error_df['hybrid_correct']
hybrid_missed_mask = kcbert_wrong_mask & ~kcbert_error_df['hybrid_correct']
hybrid_regressed_mask = kcbert_error_df['kcbert_correct'] & ~kcbert_error_df['hybrid_correct']

kcbert_hybrid_summary = pd.DataFrame([
    {'case': 'KcBERT-only 오답 전체', 'count': int(kcbert_wrong_mask.sum())},
    {'case': 'KcBERT-only 오답 중 hybrid가 정답으로 보완', 'count': int(hybrid_recovered_mask.sum())},
    {'case': 'KcBERT-only 오답 중 hybrid도 오답', 'count': int(hybrid_missed_mask.sum())},
    {'case': 'KcBERT-only 정답 중 hybrid가 오답으로 악화', 'count': int(hybrid_regressed_mask.sum())},
])

metadata_repair_summary = (
    kcbert_error_df[kcbert_wrong_mask]
    .assign(hybrid_result=np.where(hybrid_recovered_mask[kcbert_wrong_mask], 'hybrid_보완', 'hybrid_미보완'))
    .groupby('hybrid_result')
    .agg(
        count=('hybrid_result', 'size'),
        mean_photo_count=('photo_count', 'mean'),
        mean_text_length=('text_length', 'mean'),
        mean_emoji_count=('emoji_count', 'mean'),
        mean_kcbert_event_prob=('kcbert_event_prob', 'mean'),
        mean_hybrid_event_prob=('hybrid_event_prob', 'mean'),
    )
    .reset_index()
    .round(4)
)

print(f'KcBERT-only threshold: {kcbert_threshold:.4f}')
print(f'Hybrid threshold: {hybrid_threshold:.4f}')
display(kcbert_hybrid_summary)
display(metadata_repair_summary)

ambiguity_cols = [
    'raw_index',
    'store_name',
    'rating',
    'review_text',
    'cleaned_review_text',
    'photo_count',
    'text_length',
    'emoji_count',
    'actual_label',
    'kcbert_pred',
    'hybrid_pred',
    'kcbert_event_prob',
    'hybrid_event_prob',
    'kcbert_error_type',
    'hybrid_error_type',
]
ambiguity_cols = [col for col in ambiguity_cols if col in kcbert_error_df.columns]

kcbert_wrong_hybrid_correct_samples = (
    kcbert_error_df[hybrid_recovered_mask]
    .sort_values('kcbert_event_prob', ascending=False)
    [ambiguity_cols]
    .head(15)
)

kcbert_wrong_hybrid_wrong_samples = (
    kcbert_error_df[hybrid_missed_mask]
    .sort_values('kcbert_event_prob', ascending=False)
    [ambiguity_cols]
    .head(15)
)

print('KcBERT-only는 틀렸지만 hybrid가 맞춘 샘플')
display(kcbert_wrong_hybrid_correct_samples)

print('KcBERT-only와 hybrid가 모두 틀린 샘플')
display(kcbert_wrong_hybrid_wrong_samples)


## 10. 최종 결과 해석

### 10-1. 핵심 결론

이벤트 리뷰에 대한 validation F1을 기준으로 최종 모델을 선택한다. test 데이터는 선택 이후 성능 확인과 오답 분석에만 사용한다.

이는 test 성능을 직접 기준으로 모델을 선택하는 것보다 더 엄격한 기준이다.

### 10-2. 성능 비교 해석

- validation에서 F1이 가장 높은 모델을 최종 모델로 선택한다. 05번의 metadata subset ablation 모델도 같은 후보군에 포함된다.
- 선택된 모델의 test 성능을 따로 보고한다.
- 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)이 최종 선택되지 않은 경우, 현재 하이브리드 융합 전략의 한계로 직접 설명한다.

### 10-3. 메타데이터 영향도 해석

선택된 모델이 메타데이터를 직접 사용하는 경우 permutation 중요도를 해석한다. `rating`은 이후에 정제해야 할 대상이므로 입력 feature에서 제외한다.

### 10-4. 오답 데이터 해석

FP는 실제로는 일반 리뷰이지만 모델이 이벤트 리뷰로 잘못 판단한 경우다. 일반 리뷰임에도 이벤트 리뷰와 비슷한 표현이나 행동 패턴을 보여 오탐했을 가능성이 있다.

FN은 실제 이벤트 리뷰이지만 모델이 일반 리뷰로 놓친 경우다. 메타데이터 신호가 약하거나 이벤트 여부가 텍스트 안에서만 드러나는 경우일 수 있다.

KcBERT-only 오답과 hybrid 결과를 비교하면, 텍스트 임베딩만으로 어려웠던 사례에서 사진 수, 글자 수, 이모지 수 같은 행동 메타데이터가 일부 오답을 보완했는지 확인할 수 있다. 다만 이 분석은 사진 이미지 자체가 아니라 `photo_count` 기반 행동 신호만 사용한다.

### 10-5. 향후 개선 방향

가능한 개선 방식으로는 KcBERT fine-tuning, 은어/신조어 사전 보강, 문장 임베딩 모델 활용, 텍스트 특징 선택, 텍스트와 메타데이터 간 가중 융합이 있다. 현재 결과는 과장하지 않고 validation 기준 선택 결과와 test 참고 성능으로 분리해 설명한다.


## 11. 선택 모델과 제안 모델 저장

validation에서 선택된 모델은 `outputs/final_selected_model.joblib`에 저장하고, 프로젝트 제안 모델은 `outputs/final_proposed_model.joblib`에 별도로 저장한다.

07번에서 두 모델과 threshold를 그대로 사용할 수 있도록 모델, 입력 타입, feature 목록, validation 지표, test 참고 지표를 각각의 bundle로 저장한다.


In [ ]:
def make_final_bundle(bundle, model_name, row, test_row, spec, threshold, role, reason, importance):
    return {
        'model': bundle['model'],
        'model_name': model_name,
        'model_role': role,
        'feature_set': row['feature_set'],
        'input_type': spec['input_type'],
        'feature_cols': bundle.get('feature_cols'),
        'text_col': bundle.get('text_col'),
        'target_col': bundle.get('target_col', LABEL_COL),
        'default_threshold': float(bundle.get('default_threshold', 0.5)),
        'best_threshold': threshold,
        'selection_rule': 'validation F1 기준 선택 모델과 프로젝트 제안 모델을 별도 bundle로 저장',
        'selection_reason': reason,
        'validation_selected_metrics': row.to_dict(),
        'test_reference_metrics': test_row.to_dict(),
        'validation_model_comparison': validation_comparison.to_dict(orient='records'),
        'test_model_comparison': test_comparison.to_dict(orient='records'),
        'metadata_importance': importance.to_dict(orient='records'),
        'excluded_features': ['rating'],
        'next_step': '07_rating_refinement에서 이 bundle을 로드해 reviews_embeddings_extract.csv 기준 feature 행에 적용한다.',
    }


final_selected_bundle = make_final_bundle(
    selected_bundle,
    selected_model_name,
    selected_row,
    selected_test_row,
    selected_spec,
    selected_threshold,
    'selected',
    selection_reason,
    selected_importance,
)
final_proposed_bundle = make_final_bundle(
    proposed_bundle,
    proposed_model_name,
    proposed_row,
    proposed_test_row,
    proposed_spec,
    proposed_threshold,
    'proposed',
    proposed_reason,
    proposed_importance,
)

joblib.dump(final_selected_bundle, 'outputs/final_selected_model.joblib')
joblib.dump(final_proposed_bundle, 'outputs/final_proposed_model.joblib')

bundle_display_rows = []
for output_path, bundle in [
    ('outputs/final_selected_model.joblib', final_selected_bundle),
    ('outputs/final_proposed_model.joblib', final_proposed_bundle),
]:
    row = {k: v for k, v in bundle.items() if k != 'model'}
    row['output_path'] = output_path
    bundle_display_rows.append(row)

print('저장 완료: outputs/final_selected_model.joblib')
print('저장 완료: outputs/final_proposed_model.joblib')
display(pd.DataFrame(bundle_display_rows))

## 12. 07번 별점 정제와의 연결

07번은 `outputs/final_selected_model.joblib`과 `outputs/final_proposed_model.joblib`을 함께 로드해 리뷰별 이벤트 확률을 예측하고, 정제 전후 별점을 비교한다.

선택된 모델에 FP가 많으면 이벤트 리뷰를 단순히 제거하는 방식은 일반 리뷰까지 과도하게 제외할 수 있다. 그래서 07번에서는 확률 기반 가중 평균 방식도 함께 비교한다.

모델 선택과 별점 정제 단계를 분리하면 06번의 test 기반 분석이 모델 선택 과정으로 역류하는 문제를 줄일 수 있다.
